In [1]:
import pandas as pd
import numpy as np

In [2]:
"""
This script:

1. Loads your merged eye-tracking + Unity dataset.
2. Defines:
   - Vertical object size = SizeY (always)
   - Horizontal object size = max(SizeX, SizeZ) per row
3. Computes retinal image size (visual angle) for each fixation on a valid object.
4. Applies an HTC Vive Pro Eye FOV filter:
   - Excludes objects whose retinal image would not fit into the headset FOV.
5. Saves the final cleaned dataset.

ASSUMPTIONS:
- Input CSV file: "eyetracking_with_unity_and_exclude_object.csv"
- Columns present:
    - exclude_object      : 0.0 = fixation on valid object; 1.0 =  not valid
    - Eucledian_distance  : distance from eye to hit point (meters)
    - avg_dis             : distnace from eye to his point (metres)
    - SizeX               : object size along X (meters)
    - SizeY               : object size along Y (meters) -> VERTICAL size
    - SizeZ               : object size along Z (meters)
    - names (optional, just for reference in prints)
"""

# -------------------- 1. Load data --------------------
input_path = r"C:/Users/Marcel/Desktop/Study Project/VR Data/Data with Turns/combined_dataframe_mergedobj.csv"
df = pd.read_csv(input_path)
# Make sure exclude_object is numeric (0.0 / 1.0)
df["exclude_object"] = df["exclude_object"].astype(float)

C:\Users\Marcel\AppData\Local\Temp\ipykernel_1136\3989293032.py:27: DtypeWarning: Columns (68,69) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


In [3]:
# -------------------- 2. Define effective object sizes --------------------
# Vertical size is always SizeY
df["vert_size_m"] = df["SizeY"]

# Horizontal size is the larger of SizeX and SizeZ
df["horiz_size_m"] = df[["SizeX", "SizeZ"]].max(axis=1)

In [4]:
# -------------------- 3. Compute retinal image size (visual angle) --------------------
# Only compute visual angle where:
# - exclude_object == 0.0 (valid object fixation)
# - distance is present and > 0
mask = (
    (df["exclude_object"] == 0.0) &
    df["avg_dist"].notna() &
    (df["avg_dist"] > 0)
)

# Horizontal visual angle (deg) from horiz_size_m
df.loc[mask, "visual_angle_width_deg"] = 2 * np.degrees(
    np.arctan2(
        df.loc[mask, "horiz_size_m"] / 2.0,
        df.loc[mask, "avg_dist"]
    )
)

# Vertical visual angle (deg) from vert_size_m (SizeY)
df.loc[mask, "visual_angle_height_deg"] = 2 * np.degrees(
    np.arctan2(
        df.loc[mask, "vert_size_m"] / 2.0,
        df.loc[mask, "avg_dist"]
    )
)

# approximate retinal image area in deg^2
df.loc[mask, "retinal_image_area_deg2"] = (
    df.loc[mask, "visual_angle_width_deg"] *
    df.loc[mask, "visual_angle_height_deg"]
)

print("Example rows with effective sizes and computed visual angles:")
print(
    df.loc[mask, [
        "names",
        "SizeX", "SizeY", "SizeZ",
        "horiz_size_m", "vert_size_m",
        "avg_dist",
        "visual_angle_width_deg", "visual_angle_height_deg",
        "retinal_image_area_deg2"
    ]]
    .head()
)

Example rows with effective sizes and computed visual angles:
              names     SizeX    SizeY     SizeZ  horiz_size_m  vert_size_m  \
40  building01_LOD1  22.90853  15.9103  17.23831      22.90853      15.9103   
41  building01_LOD1  22.90853  15.9103  17.23831      22.90853      15.9103   
42  building01_LOD1  22.90853  15.9103  17.23831      22.90853      15.9103   
43  building01_LOD1  22.90853  15.9103  17.23831      22.90853      15.9103   
44  building01_LOD1  22.90853  15.9103  17.23831      22.90853      15.9103   

     avg_dist  visual_angle_width_deg  visual_angle_height_deg  \
40  45.835912               28.061397                19.692026   
41  45.835912               28.061397                19.692026   
42  45.835912               28.061397                19.692026   
43  45.835912               28.061397                19.692026   
44  45.835912               28.061397                19.692026   

    retinal_image_area_deg2  
40               552.585763  
41    

In [5]:
# -------------------- 4. Apply HTC Vive Pro Eye FOV filter --------------------
# Vive Pro Eye: ~110° diagonal FOV. Approximate:
H_FOV_DEG = 100.0  # approx horizontal FOV
V_FOV_DEG = 100.0  # approx vertical FOV

# Rows where we have valid visual angle and a valid object (exclude_object == 0.0)
mask_valid_angle = (
    (df["exclude_object"] == 0.0) &
    df["visual_angle_width_deg"].notna() &
    df["visual_angle_height_deg"].notna()
)

# Does the object fit inside the (approximate) HMD FOV?
df.loc[mask_valid_angle, "fits_horiz_fov"] = (
    df.loc[mask_valid_angle, "visual_angle_width_deg"] <= H_FOV_DEG
)
df.loc[mask_valid_angle, "fits_vert_fov"] = (
    df.loc[mask_valid_angle, "visual_angle_height_deg"] <= V_FOV_DEG
)
df.loc[mask_valid_angle, "fits_full_fov"] = (
    df.loc[mask_valid_angle, "fits_horiz_fov"] &
    df.loc[mask_valid_angle, "fits_vert_fov"]
)

# Exclude objects that do NOT fit the FOV:
# set exclude_object = 1.0 for them
df.loc[
    mask_valid_angle & (~df["fits_full_fov"].fillna(False)),
    "exclude_object"
] = 1.0

C:\Users\Marcel\AppData\Local\Temp\ipykernel_1136\2910599848.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask_valid_angle & (~df["fits_full_fov"].fillna(False)),


In [6]:
cols_to_drop = [
    "Unnamed: 0",
    "ordinalOfHit",
    "hitObjectColliderName",
    "Euclidean_distance",
    "Collider_Categorical",
    "Continuous_Time",
    "Bitmask_flag",
    "Interpolated_collider",
    "Collider_shift",
    "counter",
    "Time_of_Gaze",
    "Gaze",
    "distance",
    "Collider_CategoricalN",
    "fits_horiz_fov",
    "fits_vert_fov",
    "fits_full_fov",
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

In [7]:
# ---------------------------------------------------------
# Mark additional scene-spanning objects as excluded
# Only for fixation rows (fixation_id not NA)
# Uses the column "names"
# ---------------------------------------------------------

name_norm = (
    df["names"]
    .astype(str)
    .str.strip()
    .str.lower()
)

fix_mask = df["fixation_id"].notna()

# Exact names to exclude
exclude_exact = {
    "arabic_v1_ceiling",
    "arabic_v1_floor",
    "e_plateau.001",
    "euro_v6_1_forged_fence",
    "euro_v7_3_roof_floor",
    "barbwire0",
    "basketballcourt_58",
    "complete_fence.004",
    "concrete_fence_v1_l2",
    "concrete_fence_v1_s1",
    "concrete_fence_v1_s1 (1)",
    "concrete_fence_v1_s2 (37)",
    "concrete_fence_v2_end",
    "concrete_fence_v2_end (1)",
    "concrete_fence_v2_l",
    "concrete_fence_v2_lattice_l",
    "concrete_fence_v2_lattice_s",
    "concrete_fence_v2_s",
    "cyclone0",
    "e_road.001",
    "fence",
    "fence.001",
    "fence0",
    "fence9",
    "fence_0",
    "fence_27",
    "fence_7",
    "fencing",
    "field",
    "field_e.004",
    "forged_fence_v1",
    "gras",
    "grayroom",
    "hedge",
    "hedge_15",
    "hedge_2",
    "maraz_cafe_place.001",
    "moss.003",
    "o_road.001",
    "posts0",
    "pavement_a.002",
    "pavement_e.010",
    "pavement_ha.002",
    "pavement_t01.002",
    "pavement_ua4.001",
    "pavement_vb.004",
    "pavement_w_fix.001",
    "raods_s_fix.009",
    "road.009",
    "road.014",
    "road_base_network.004",
    "road_dense_housing.008",
    "road_dense_housing.011",
    "road_green_area.007",
    "road_playground.002",
    "scaffold_base_02_lod0",
    "scaffold_topboard_01_lod0",
    "terrain_b.002",
    "terrain_f.001",
    "terrain_u_lp.001",
    "v_new_roads.002",
    "w_new_roads.001",
}

# Broad prefixes to exclude
exclude_prefixes = (
    "pavement",
    "road",
    "terrain",
    "fence",
    "hedge",
    "tree_planters_v1",
    "broadleaf_mobile",
    "wall",
    "oldcitywall",
    "smallwall",
)

# Exact exceptions to KEEP
keep_exact = {
    "euro_v2_market_v3",
    "road_block_v1",
    "road_housing_suburb.003",
    "smallwall_38",
    "smallwall_12",
    "smallwall_17",
    "smallwall_19",
    "smallwall_45",
    "smallwall_39",
}

# Prefix exceptions to KEEP
keep_prefixes = (
    "tree_planters_v1_4lr",
    "tree_planters_v1_4ls",
)

# Start with exact-name exclusions
exclude_mask = name_norm.isin(exclude_exact)

# Add broad prefix-based exclusions
exclude_mask |= name_norm.str.startswith(exclude_prefixes)

# Remove exact exceptions
exclude_mask &= ~name_norm.isin(keep_exact)

# Remove prefix exceptions
for kp in keep_prefixes:
    exclude_mask &= ~name_norm.str.startswith(kp)

# Apply only to fixation rows
final_exclude_mask = fix_mask & exclude_mask

df.loc[final_exclude_mask, "exclude_object"] = 1.0

print("Rows marked excluded by manual scene-spanning filter:", int(final_exclude_mask.sum()))
print("\nExample excluded names:")
print(
    df.loc[final_exclude_mask, "names"]
      .dropna()
      .astype(str)
      .drop_duplicates()
      .sort_values()
      .head(150)
      .tolist()
)

Rows marked excluded by manual scene-spanning filter: 610645

Example excluded names:
['Arabic_v1_ceiling', 'Arabic_v1_floor', 'BasketballCourt_58', 'Broadleaf_Mobile (12)', 'Broadleaf_Mobile (176)', 'Broadleaf_Mobile (206)', 'Broadleaf_Mobile (4)', 'Broadleaf_Mobile (9)', 'Complete_fence.004', 'Concrete_fence_v1_L2', 'Concrete_fence_v1_s1', 'Concrete_fence_v1_s1 (1)', 'Concrete_fence_v1_s2 (37)', 'Concrete_fence_v2_L', 'Concrete_fence_v2_S', 'Concrete_fence_v2_lattice_L', 'Concrete_fence_v2_lattice_S', 'E_plateau.001', 'E_road.001', 'Euro_v6_1_Forged_fence', 'Euro_v7_3_Roof_floor', 'Fence', 'Fence.001', 'Fence0', 'Fence1', 'Fence2', 'Fence3', 'Fence4', 'Fence5', 'Fence6', 'Fence7', 'Fence8', 'Fence9', 'Fence_0', 'Fence_1', 'Fence_10', 'Fence_11', 'Fence_12', 'Fence_13', 'Fence_14', 'Fence_15', 'Fence_16', 'Fence_18', 'Fence_19', 'Fence_2', 'Fence_20', 'Fence_21', 'Fence_22', 'Fence_23', 'Fence_24', 'Fence_25', 'Fence_26', 'Fence_27', 'Fence_28', 'Fence_29', 'Fence_3', 'Fence_30', 'Fen

In [8]:
# -------------------- 5. Report and save --------------------
total_rows = len(df)
n_valid_before = int(mask.sum())
n_valid_after = int((df["exclude_object"] == 0.0).sum())

print("\n--- Summary ---")
print(f"Total rows in dataset: {total_rows}")
print(f"Rows with valid objects and distance (before FOV filter): {n_valid_before}")
print(f"Rows with exclude_object == 0.0 after FOV filter: {n_valid_after}")

output_path = r"C:\Users\Marcel\Desktop\Study Project\VR Data\Data with Turns\combined_dataframe_obj_fovfiltered.csv"
df.to_csv(output_path, index=False)
print(f"\nSaved updated dataset with visual angles and FOV filter to:\n{output_path}")


--- Summary ---
Total rows in dataset: 3270015
Rows with valid objects and distance (before FOV filter): 1166146
Rows with exclude_object == 0.0 after FOV filter: 1165710

Saved updated dataset with visual angles and FOV filter to:
C:\Users\Marcel\Desktop\Study Project\VR Data\Data with Turns\combined_dataframe_obj_fovfiltered.csv
